# **TITLE**
# Analysis of Trading Behavior using Fear & Greed Index

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# **OBJECTIVE**

The objective of this project is to analyze the relationship between market sentiment
(Fear vs Greed) and trading behavior.

We aim to understand how sentiment affects:
- Profit (PnL)
- Win rate
- Trading activity

This helps in understanding behavioral patterns in financial markets.

# **DATASET DESCRIPTION**

Two datasets are used:

1. Fear & Greed Index Dataset:
   - Contains daily market sentiment (Fear/Greed)
   - Columns: date, value, classification

2. Trading Data:
   - Contains individual trade records
   - Columns include: Account, Execution Price, Size USD, Closed PnL, Timestamp

The datasets are merged based on date to analyze sentiment vs trading behavior.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
from google.colab import files
files.upload()

In [ ]:
sentiment = pd.read_csv("fear_greed_index.csv")
trades = pd.read_csv("historical_data.csv")

print("Sentiment Shape:", sentiment.shape)
print("Trades Shape:", trades.shape)

In [ ]:
print("\nMissing Values:\n")
print(sentiment.isnull().sum())
print(trades.isnull().sum())

# Drop duplicates
sentiment.drop_duplicates(inplace=True)
trades.drop_duplicates(inplace=True)

# DATA PREPROCESSING

- Converted timestamps into datetime format
- Extracted date from timestamp
- Cleaned text (removed 'IST' from timestamp)
- Handled missing values using 'coerce'
- Merged trading data with sentiment data on 'date'

In [ ]:
trades['Timestamp IST'] = trades['Timestamp IST'].astype(str).str.replace('IST', '')

trades['Timestamp IST'] = pd.to_datetime(trades['Timestamp IST'], errors='coerce')

trades['date'] = trades['Timestamp IST'].dt.date

sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

In [ ]:
print(trades['date'].head())
print(sentiment['date'].head())

In [ ]:
print(sentiment.columns)
print(trades.columns)

# FEATURE ENGINEERING

New features were created:

- win: Whether a trade resulted in profit (Closed PnL > 0)
- daily_pnl: Total profit per account per day
- win_rate: Average success rate of trades
- trade_count: Number of trades per day

These features help in analyzing trading behavior effectively.

In [ ]:
df = trades.merge(sentiment, on='date', how='left')

print(df.head())

In [ ]:
# Win/Loss
df['win'] = df['Closed PnL'] > 0

# Sentiment encoding
df['sentiment_num'] = df['classification'].map({'Fear': 0, 'Greed': 1})

# Daily metrics
metrics = df.groupby(['Account', 'date']).agg({
    'Closed PnL': 'sum',
    'Execution Price': 'mean',
    'Size USD': 'mean',
    'Trade ID': 'count',
    'win': 'mean'
}).reset_index()

metrics.rename(columns={
    'Closed PnL': 'daily_pnl',
    'win': 'win_rate',
    'Trade ID': 'trade_count'
}, inplace=True)

metrics = metrics.merge(sentiment, on='date', how='left')

metrics.head()

In [ ]:
df['win'] = df['Closed PnL'] > 0

metrics = df.groupby(['Account', 'date']).agg({
    'Closed PnL': 'sum',
    'win': 'mean'
}).reset_index()

metrics.rename(columns={'Closed PnL': 'daily_pnl', 'win': 'win_rate'}, inplace=True)

metrics = metrics.merge(sentiment[['date', 'classification']], on='date', how='left')

print(metrics.groupby('classification')['daily_pnl'].mean())
print(metrics.groupby('classification')['win_rate'].mean())

In [ ]:
print("\nAverage PnL by Sentiment:")
print(metrics.groupby('classification')['daily_pnl'].mean())

print("\nWin Rate by Sentiment:")
print(metrics.groupby('classification')['win_rate'].mean())

In [ ]:
print("\nTrade Count by Sentiment:")
print(metrics.groupby('classification')['trade_count'].mean())

# VISUALISATION

The following visualizations were used:

- Boxplot: Distribution of PnL by sentiment
- Bar plot: Win rate comparison
- Bar plot: Trade count comparison
- Heatmap: Correlation between features

These visualizations help in understanding patterns clearly.

In [ ]:
corr = metrics[['daily_pnl','win_rate','trade_count']].corr()

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.heatmap(corr, annot=True)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# High vs Low trade activity
threshold = metrics['trade_count'].median()

metrics['activity_group'] = metrics['trade_count'].apply(
    lambda x: 'High' if x > threshold else 'Low'
)

seg_result = metrics.groupby(['activity_group', 'classification'])['daily_pnl'].mean()

print(seg_result)

In [ ]:
# Trade activity comparison
plt.figure()
sns.barplot(x='classification', y='trade_count', data=metrics)
plt.title("Trade Count by Sentiment")
plt.show()

In [ ]:
# Win rate comparison
plt.figure()
sns.barplot(x='classification', y='win_rate', data=metrics)
plt.title("Win Rate by Sentiment")
plt.show()

In [ ]:
trades['Timestamp'] = pd.to_datetime(trades['Timestamp'], errors='coerce')
trades['date'] = trades['Timestamp'].dt.date

In [ ]:
print(trades['date'].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.boxplot(x='classification', y='daily_pnl', data=metrics)
plt.title("PnL Distribution by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Daily PnL")
plt.show()

In [ ]:
plt.figure()
sns.violinplot(x='classification', y='daily_pnl', data=metrics)
plt.title("PnL Distribution Shape")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Create target variable
metrics['profit_flag'] = (metrics['daily_pnl'] > 0).astype(int)

# Features and target
X = metrics[['win_rate', 'trade_count']]
y = metrics['profit_flag']

# Remove missing values
X = X.fillna(0)
y = y.fillna(0)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

A simple Logistic Regression model is used to predict whether a trading day
results in profit or loss.

Features used:
- Win rate
- Trade count

The model is trained on historical data and evaluated using accuracy.

In [ ]:
import seaborn as sns

sns.heatmap(pd.crosstab(y_test, y_pred), annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# KEY INSIGHTS

- Traders tend to trade more during Greed phases
- Fear phases show reduced trading activity
- Profitability varies with sentiment conditions
- Behavioral patterns play a key role in trading outcomes

# CONCLUSION

This analysis shows that market sentiment significantly influences trading behavior.

- Greed leads to higher activity and risk-taking
- Fear leads to cautious trading
- Performance metrics like PnL and win rate vary with sentiment

Understanding these patterns can help in making better trading decisions.